In [1]:
import os
import sys
import glob
import re
import logging  
import h5py
import numpy as np
import pandas as pd  
import matplotlib.pyplot as plt
import tensorflow as tf

# --- Suppress Matplotlib Debug Output ---
logging.getLogger('matplotlib').setLevel(logging.WARNING)
logging.getLogger('matplotlib.ticker').setLevel(logging.WARNING)

# --- Path Setups (Adjust as needed) ---
qmc_algo_path = os.path.abspath('/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/qmc_algo_tools') 
dev = os.path.abspath('/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/developer_tools') 
sys.path.insert(0, qmc_algo_path) 
sys.path.insert(0, dev) 

# =====================================================================
# --- Base and Model Classes (Included directly for dependency) ---
# =====================================================================

# Simulated base class template if not explicitly loaded from tools
class DensitySmoothFitBase:
    def __init__(self, density_ref, **kw):
        self.density_ref = density_ref
        
    def denoise_transformed(self, dq):
        return self.smoothed_fit(dq)

    def smoothed_fit(self, dq):
        raise NotImplementedError

class DensitySmoothFitLocalLine(DensitySmoothFitBase):
    def __init__(self, degree=3, window=4, **kw):
        self.degree = degree
        self.window = window
        DensitySmoothFitBase.__init__(self, **kw)

    def pad(self, d):
        return d

    def unpad(self, d):
        return d

    def smoothed_fit(self, dq):
        nx, ny, nz = dq.shape
        n = self.degree
        w = self.window
        ww = 2 * w + 1
        dqs = np.zeros(dq.shape)
        indf = np.arange(ww, dtype=float)
        
        # smooth over z
        for i in range(nx):
            for j in range(ny):
                ind = (np.arange(ww) - w) % nz
                for k in range(nz):
                    dv = dq[i, j, ind]
                    p = np.polyfit(indf, dv, n)
                    dqs[i, j, k] = np.polyval(p, indf[w])
                    ind = (ind + 1) % nz
        # smooth over y
        for i in range(nx):
            for k in range(nz):
                ind = (np.arange(ww) - w) % ny
                for j in range(ny):
                    dv = dqs[i, ind, k]
                    p = np.polyfit(indf, dv, n)
                    dqs[i, j, k] = np.polyval(p, indf[w])
                    ind = (ind + 1) % ny
        # smooth over x
        for j in range(ny):
            for k in range(nz):
                ind = (np.arange(ww) - w) % nx
                for i in range(nx):
                    dv = dqs[ind, j, k]
                    p = np.polyfit(indf, dv, n)
                    dqs[i, j, k] = np.polyval(p, indf[w])
                    ind = (ind + 1) % nx
        return dqs

# =====================================================================
# --- Helper Functions & Metric Calculations ---
# =====================================================================

def D_JS(p1, p2, tol=1e-16):
    p1 = p1 / np.sum(p1)
    p2 = p2 / np.sum(p2)
    pm = (p1 + p2) / 2
    p1_nonzero = np.abs(p1) > tol
    p2_nonzero = np.abs(p2) > tol
    p1  = np.abs(p1[p1_nonzero])
    pm1 = np.abs(pm[p1_nonzero])
    p2  = np.abs(p2[p2_nonzero])
    pm2 = np.abs(pm[p2_nonzero])
    d = .5 * ( (p1 * np.log(p1 / pm1)).sum() + (p2 * np.log(p2 / pm2)).sum() )
    d /= np.log(2)
    return d

def transform(density, density_ref, transform_type):
    if transform_type == 'residual_noise':
        return (density - density_ref) / np.sqrt(np.abs(density_ref))
    raise RuntimeError('bad transform type')

def inverse_transform(density_trans, density_ref, transform_type):
    if transform_type == 'residual_noise':
        return density_ref + np.sqrt(np.abs(density_ref)) * density_trans
    raise RuntimeError('bad transform type')

# =====================================================================
# --- Main Search Routine ---
# =====================================================================

if __name__ == "__main__":
    # --- 1. Load Reference Data ---
    dmc_ref_path = '/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/dmc_J2/density_tot_ref_mean.h5'
    with h5py.File(dmc_ref_path, 'r') as file:
        ref_d = file['density'][:]
    print(f"dmc Ref shape: {ref_d.shape}")

    dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'
    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]

    # --- 2. Track & Sort Files to Find the Middle Sample Count ---
    data_dir = '/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/dmc_J2/'
    file_pattern = os.path.join(data_dir, "density_tot_dmc_mix_mean_*.h5")
    files = glob.glob(file_pattern)

    file_data = []
    for fpath in files:
        match = re.search(r'mean_(\d+)\.h5', os.path.basename(fpath))
        if match:
            file_data.append({'samples': int(match.group(1)), 'path': fpath})

    if not file_data:
        raise FileNotFoundError(f"No files matching the pattern found in {data_dir}")

    # Convert to DataFrame, sort by sample count, and grab the absolute median entry
    df_files = pd.DataFrame(file_data).sort_values(by='samples').reset_index(drop=True)
    middle_index = len(df_files) // 2
    target_file = df_files.iloc[middle_index]['path']
    target_samples = df_files.iloc[middle_index]['samples']

    print(f"\nFound {len(df_files)} available files.")
    print(f"Targeting single middle sample configuration with {target_samples} samples.")
    print(f"Processing File: {target_file}\n")

    # --- 3. Instantiate Hyperparameter Grid Models ---
    degrees_to_test = [1, 2, 3]
    windows_to_test = [2, 3, 4, 5]
    trans_type = 'residual_noise'

    models = {}
    for deg in degrees_to_test:
        for win in windows_to_test:
            models[(deg, win)] = DensitySmoothFitLocalLine(density_ref=dft_d, degree=deg, window=win)

    # --- 4. Execution Loop on the Middle Sample ---
    with h5py.File(target_file, 'r') as f:
        test_d = np.array(f['density'])

    # Log foundational metric
    file_results = {
        "Samples": target_samples,
        "Noisy_dmc_JS": D_JS(test_d, ref_d)
    }

    # Evaluate each grid state
    for (deg, win), dm in models.items():
        # --- Run WITHOUT Transformation ---
        denoised_raw = dm.smoothed_fit(test_d)
        js_raw = D_JS(denoised_raw, ref_d)
        file_results[f"Raw_deg{deg}_win{win}"] = js_raw

        # --- Run WITH Transformation ---
        trans_d = transform(test_d, dft_d, trans_type)
        denoised_trans_eval = dm.smoothed_fit(trans_d)
        denoised_trans_inv = inverse_transform(denoised_trans_eval, dft_d, trans_type)
        
        # Optional: Clamp to positive bounds if physically required by your application
        # denoised_trans_inv = np.maximum(denoised_trans_inv, 0.0)
        
        js_trans = D_JS(denoised_trans_inv, ref_d)
        file_results[f"Trans_deg{deg}_win{win}"] = js_trans

    # --- 5. Compile and Analyze Leaderboard ---
    df_results = pd.DataFrame([file_results])
    scores = df_results.drop(columns=["Samples", "Noisy_dmc_JS"]).iloc[0].sort_values()

    print("=========================================================")
    print("      HYPERPARAMETER SEARCH LEADERBOARD (D_JS)")
    print("=========================================================")
    print(f"Baseline Noisy D_JS: {file_results['Noisy_dmc_JS']:.6e}\n")
    print(scores.to_string())
    print("=========================================================")
    print(f"Best Setup Found: {scores.index[0]} (D_JS: {scores.iloc[0]:.6e})")
    print("=========================================================")

2026-06-07 09:12:44.443288: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780848764.459337    9448 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780848764.464929    9448 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780848764.478797    9448 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780848764.478809    9448 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780848764.478811    9448 computation_placer.cc:177] computation placer alr

dmc Ref shape: (64, 64, 64)

Found 33 available files.
Targeting single middle sample configuration with 10460209 samples.
Processing File: /pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/dmc_J2/density_tot_dmc_mix_mean_0010460209.h5

      HYPERPARAMETER SEARCH LEADERBOARD (D_JS)
Baseline Noisy D_JS: 6.217853e-04

Trans_deg1_win5    0.000015
Trans_deg1_win4    0.000016
Trans_deg1_win3    0.000019
Trans_deg1_win2    0.000027
Trans_deg2_win5    0.000034
Trans_deg3_win5    0.000034
Trans_deg2_win4    0.000044
Trans_deg3_win4    0.000044
Raw_deg3_win4      0.000062
Raw_deg2_win4      0.000062
Trans_deg2_win3    0.000062
Trans_deg3_win3    0.000062
Raw_deg2_win3      0.000066
Raw_deg3_win3      0.000066
Raw_deg3_win5      0.000086
Raw_deg2_win5      0.000086
Trans_deg2_win2    0.000115
Trans_deg3_win2    0.000115
Raw_deg3_win2      0.000115
Raw_deg2_win2      0.000115
Raw_deg1_win2      0.000291
Raw_deg1_win3      0.000849
Raw_deg1_win4      0.001894
Raw_deg1_win5      0.0

In [2]:
print("here")

here


In [3]:
import os
import sys
import glob
import re
import logging  
import h5py
import numpy as np
import pandas as pd  
import tensorflow as tf

# --- Suppress Debug Output ---
logging.getLogger('matplotlib').setLevel(logging.WARNING)
logging.getLogger('matplotlib.ticker').setLevel(logging.WARNING)

# --- Path Setups & Codebase Imports ---
qmc_algo_path = os.path.abspath('/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/qmc_algo_tools') 
dev = os.path.abspath('/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/developer_tools') 
sys.path.insert(0, qmc_algo_path) 
sys.path.insert(0, dev) 

# Importing the SN wrapper since the target hyperparameters map to smoothn
from qmc_algo_tools.density_denoise import DensitySmoothFitSN 

# =====================================================================
# --- Helper Functions & Metric Calculations ---
# =====================================================================

def D_JS(p1, p2, tol=1e-16):
    p1 = p1 / np.sum(p1)
    p2 = p2 / np.sum(p2)
    pm = (p1 + p2) / 2
    p1_nonzero = np.abs(p1) > tol
    p2_nonzero = np.abs(p2) > tol
    p1  = np.abs(p1[p1_nonzero])
    pm1 = np.abs(pm[p1_nonzero])
    p2  = np.abs(p2[p2_nonzero])
    pm2 = np.abs(pm[p2_nonzero])
    d = .5 * ( (p1 * np.log(p1 / pm1)).sum() + (p2 * np.log(p2 / pm2)).sum() )
    d /= np.log(2)
    return d

def transform(density, density_ref, transform_type):
    if transform_type == 'residual_noise':
        return (density - density_ref) / np.sqrt(np.abs(density_ref))
    raise RuntimeError('bad transform type')

def inverse_transform(density_trans, density_ref, transform_type):
    if transform_type == 'residual_noise':
        return density_ref + np.sqrt(np.abs(density_ref)) * density_trans
    raise RuntimeError('bad transform type')

# =====================================================================
# --- Main Search Routine ---
# =====================================================================

if __name__ == "__main__":
    # --- 1. Load Reference Data ---
    dmc_ref_path = '/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/dmc_J2/density_tot_ref_mean.h5'
    with h5py.File(dmc_ref_path, 'r') as file:
        ref_d = file['density'][:]
    print(f"dmc Ref shape: {ref_d.shape}")

    dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'
    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]

    # --- 2. Track & Sort Files to Find the Middle Sample Count ---
    data_dir = '/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/dmc_J2/'
    file_pattern = os.path.join(data_dir, "density_tot_dmc_mix_mean_*.h5")
    files = glob.glob(file_pattern)

    file_data = []
    for fpath in files:
        match = re.search(r'mean_(\d+)\.h5', os.path.basename(fpath))
        if match:
            file_data.append({'samples': int(match.group(1)), 'path': fpath})

    if not file_data:
        raise FileNotFoundError(f"No files matching the pattern found in {data_dir}")

    # Convert to DataFrame, sort by sample count, and grab the absolute median entry
    df_files = pd.DataFrame(file_data).sort_values(by='samples').reset_index(drop=True)
    middle_index = len(df_files) // 2
    target_file = df_files.iloc[middle_index]['path']
    target_samples = df_files.iloc[middle_index]['samples']

    print(f"\nFound {len(df_files)} available files.")
    print(f"Targeting single middle sample configuration with {target_samples} samples.")
    print(f"Processing File: {target_file}\n")

    # --- 3. Instantiate Hyperparameter Grid Models (DensitySmoothFitSN) ---
    # Define the parameter space based on smoothn parameters
    smooth_vals = [None, 0.2, 0.5, 0.8]        # Float 0-1, or None for automatic
    order_vals  = [1, 2]                    # Allowed smoothing exponents
    method_vals = ['cauchy', 'talworth', 'bisquare'] # Weight computation
    trans_type  = 'residual_noise'

    models = {}
    for s in smooth_vals:
        for o in order_vals:
            for w in method_vals:
                # SAFEGUARD: The GCV optimizer crashes if smooth is Auto (None) and order is 0
                if s is None and o == 0:
                    continue 
                
                models[(s, o, w)] = DensitySmoothFitSN(
                    density_ref=dft_d, 
                    smooth=s, 
                    order=o, 
                    weight=w
                )

    # --- 4. Execution Loop on the Middle Sample ---
    with h5py.File(target_file, 'r') as f:
        test_d = np.array(f['density'])

    # Log foundational metric
    file_results = {
        "Samples": target_samples,
        "Noisy_dmc_JS": D_JS(test_d, ref_d)
    }

    # Evaluate each grid state
    print("Running grid search. This may take a moment depending on grid size...")
    for (s, o, w), dm in models.items():
        # Clean up labels for Pandas reporting
        s_label = "Auto" if s is None else f"{s}"
        config_name = f"S:{s_label}_O:{o}_W:{w}"

        # --- Run WITHOUT Transformation ---
        denoised_raw = dm.smoothed_fit(test_d)
        js_raw = D_JS(denoised_raw, ref_d)
        file_results[f"Raw_{config_name}"] = js_raw

        # --- Run WITH Transformation ---
        trans_d = transform(test_d, dft_d, trans_type)
        denoised_trans_eval = dm.smoothed_fit(trans_d)
        denoised_trans_inv = inverse_transform(denoised_trans_eval, dft_d, trans_type)
        
        js_trans = D_JS(denoised_trans_inv, ref_d)
        file_results[f"Trans_{config_name}"] = js_trans

    # --- 5. Compile and Analyze Leaderboard ---
    df_results = pd.DataFrame([file_results])
    scores = df_results.drop(columns=["Samples", "Noisy_dmc_JS"]).iloc[0].sort_values()

    print("\n=========================================================")
    print("      HYPERPARAMETER SEARCH LEADERBOARD (D_JS)")
    print("=========================================================")
    print(f"Baseline Noisy D_JS: {file_results['Noisy_dmc_JS']:.6e}\n")
    print(scores.head(15).to_string()) # Print top 15 results
    print("=========================================================")
    print(f"Best Setup Found: {scores.index[0]} \n(D_JS: {scores.iloc[0]:.6e})")
    print("=========================================================")

dmc Ref shape: (64, 64, 64)

Found 33 available files.
Targeting single middle sample configuration with 10460209 samples.
Processing File: /pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/dmc_J2/density_tot_dmc_mix_mean_0010460209.h5

Running grid search. This may take a moment depending on grid size...


/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/qmc_algo_tools/qmc_algo_tools/smoothn_pytesmo.py:979: UserWarning: [0.00186684]: the LOWER bound for smooth has been reached. Put smooth as an input variable if required.
  warnings.warn(



      HYPERPARAMETER SEARCH LEADERBOARD (D_JS)
Baseline Noisy D_JS: 6.217853e-04

Trans_S:Auto_O:1_W:cauchy      0.000015
Trans_S:Auto_O:1_W:bisquare    0.000016
Trans_S:Auto_O:2_W:cauchy      0.000024
Trans_S:Auto_O:2_W:bisquare    0.000025
Trans_S:0.8_O:1_W:cauchy       0.000039
Trans_S:Auto_O:2_W:talworth    0.000040
Trans_S:0.8_O:2_W:cauchy       0.000040
Trans_S:0.8_O:1_W:bisquare     0.000041
Trans_S:0.8_O:2_W:bisquare     0.000041
Trans_S:0.8_O:2_W:talworth     0.000045
Trans_S:0.5_O:2_W:cauchy       0.000046
Trans_S:Auto_O:1_W:talworth    0.000047
Trans_S:0.5_O:2_W:bisquare     0.000047
Trans_S:0.5_O:1_W:cauchy       0.000052
Trans_S:0.8_O:1_W:talworth     0.000052
Best Setup Found: Trans_S:Auto_O:1_W:cauchy 
(D_JS: 1.537334e-05)
